---
title: "Chapter -- Support Vector Machines"
jupyter: python3

execute: 
  enabled: true
---

{{< chapter-actions >}}

## Introduction

Support Vector Machines (SVMs) can be used for classification, regression, and novelty detection. Their central idea is geometric: in the linearly separable hard-margin setting, an SVM selects the separating hyperplane with the largest geometric margin. Practical soft-margin SVMs instead balance margin width against training violations [@cortes1995svm].

SVMs are especially effective for small- and medium-sized datasets with complex decision boundaries. They can model both linear and nonlinear relationships, but they are sensitive to feature scaling and their performance depends strongly on the choice of hyperparameters.

::: {.callout-note}
## Learning objectives

After completing this chapter, you should be able to:

- explain the concepts of separating hyperplane, margin, and support vectors;
- distinguish hard-margin and soft-margin classification;
- interpret the hyperparameter `C`;
- explain why feature scaling is essential for SVMs;
- use linear, polynomial, and radial basis function kernels;
- interpret the role of `gamma` in an RBF kernel;
- fit SVM classifiers and regressors with Scikit-Learn;
- apply a one-class SVM for novelty detection;
- diagnose underfitting and overfitting in an SVM model.
:::

In [ ]:
#| label: chapter05-imports
#| include: false

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris, make_moons
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.svm import LinearSVC, OneClassSVM, SVC, LinearSVR, SVR

## Linear SVM classification

Consider a binary classification problem with a feature vector

$$
\mathbf{x} =
\begin{bmatrix}
x_1 \\
x_2 \\
\vdots \\
x_p
\end{bmatrix}.
$$

A linear classifier uses a decision function of the form

$$
f(\mathbf{x}) = \mathbf{w}^{\mathsf T}\mathbf{x} + b,
$$

where $\mathbf{w}$ contains the model coefficients and $b$ is the intercept.

The decision boundary is the hyperplane

$$
\mathbf{w}^{\mathsf T}\mathbf{x} + b = 0.
$$

For binary labels $y \in \{-1,+1\}$, the predicted class is determined by the sign of the decision function:

$$
\widehat{y} =
\begin{cases}
+1, & f(\mathbf{x}) \geq 0,\\
-1, & f(\mathbf{x}) < 0.
\end{cases}
$$

In two dimensions, the decision boundary is a line. In three dimensions, it is a plane. In higher-dimensional spaces, it is called a hyperplane.

In [ ]:
#| label: fig-large-margin-classification
#| fig-cap: Several boundaries separate the classes; a finite-C linear SVM closely approximates the hard-margin separator.
#| code-fold: true
#| code-summary: Show code

from sklearn import datasets
from sklearn.svm import SVC

# Load the Iris dataset
iris = datasets.load_iris(as_frame=True)

X_margin = iris.data[
    ["petal length (cm)", "petal width (cm)"]
].to_numpy()

y_margin = iris.target.to_numpy()

# Keep only Iris setosa and Iris versicolor
binary_mask = (y_margin == 0) | (y_margin == 1)

X_margin = X_margin[binary_mask]
y_margin = y_margin[binary_mask]

# Finite C remains a soft-margin model. Here C=100 closely approximates the
# hard-margin solution for these separable, consistently scaled features.
svm_margin_clf = SVC(
    kernel="linear",
    C=100
)

svm_margin_clf.fit(
    X_margin,
    y_margin
)


def plot_svc_decision_boundary(model, xmin, xmax, ax):
    """
    Plot the decision boundary, margins, and support vectors
    of a fitted linear SVC model.
    """
    w = model.coef_[0]
    b = model.intercept_[0]

    x0 = np.linspace(xmin, xmax, 200)

    # Decision boundary:
    # w[0] * x0 + w[1] * x1 + b = 0
    decision_boundary = (
        -w[0] / w[1] * x0
        - b / w[1]
    )

    # Margin boundaries:
    # w^T x + b = ±1
    margin = 1 / abs(w[1])

    margin_upper = decision_boundary + margin
    margin_lower = decision_boundary - margin

    support_vectors = model.support_vectors_

    ax.plot(
        x0,
        decision_boundary,
        linewidth=2,
        label="SVM decision boundary"
    )

    ax.plot(
        x0,
        margin_upper,
        "--",
        linewidth=1.5
    )

    ax.plot(
        x0,
        margin_lower,
        "--",
        linewidth=1.5
    )

    ax.scatter(
        support_vectors[:, 0],
        support_vectors[:, 1],
        s=180,
        facecolors="none",
        linewidths=1.5,
        label="Support vectors"
    )


# Three alternative separating boundaries
x_grid = np.linspace(0, 5.5, 200)

separator_1 = 5 * x_grid - 20
separator_2 = x_grid - 1.8
separator_3 = 0.1 * x_grid + 0.5

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 4),
    sharey=True
)

# ---------------------------------------------------------
# Left panel: several valid but arbitrary separators
# ---------------------------------------------------------

ax = axes[0]

ax.plot(
    x_grid,
    separator_1,
    "--",
    linewidth=2,
    label="Possible separator"
)

ax.plot(
    x_grid,
    separator_2,
    linewidth=2
)

ax.plot(
    x_grid,
    separator_3,
    linewidth=2
)

ax.plot(
    X_margin[y_margin == 1, 0],
    X_margin[y_margin == 1, 1],
    "s",
    label="Iris versicolor"
)

ax.plot(
    X_margin[y_margin == 0, 0],
    X_margin[y_margin == 0, 1],
    "o",
    label="Iris setosa"
)

ax.set_xlabel("Petal length (cm)")
ax.set_ylabel("Petal width (cm)")
ax.set_xlim(0, 5.5)
ax.set_ylim(0, 2)
ax.set_aspect("equal")
ax.set_title("Possible linear separators")
ax.grid(alpha=0.3)
ax.legend(loc="upper left", fontsize=8)

# ---------------------------------------------------------
# Right panel: soft-margin approximation to the hard-margin separator
# ---------------------------------------------------------

ax = axes[1]

plot_svc_decision_boundary(
    svm_margin_clf,
    xmin=0,
    xmax=5.5,
    ax=ax
)

ax.plot(
    X_margin[y_margin == 1, 0],
    X_margin[y_margin == 1, 1],
    "s",
    label="Iris versicolor"
)

ax.plot(
    X_margin[y_margin == 0, 0],
    X_margin[y_margin == 0, 1],
    "o",
    label="Iris setosa"
)

ax.set_xlabel("Petal length (cm)")
ax.set_xlim(0, 5.5)
ax.set_ylim(0, 2)
ax.set_aspect("equal")
ax.set_title("Near hard-margin solution")
ax.grid(alpha=0.3)
ax.legend(loc="upper left", fontsize=8)

plt.tight_layout()
plt.show()

The left panel shows that several linear decision boundaries can separate the two classes correctly. However, these boundaries are not equally robust: some of them pass very close to the training observations, so a small change in the data could lead to misclassification.

The right panel shows a finite-`C` soft-margin solution that approximates the hard-margin separator for this separable dataset. In exact hard-margin classification, the SVM maximizes the minimum geometric distance to the training observations; with finite `C`, it trades that goal against margin violations.

The dashed lines represent the margins, while the highlighted observations are the support vectors. These support vectors determine the position and orientation of the separating hyperplane.


### Margin and support vectors

Many different hyperplanes may separate a linearly separable dataset. Because multiplying $(\mathbf{w},b)$ by a positive constant leaves the decision boundary unchanged, hard-margin SVMs use the **canonical normalization**

$$
\min_i y_i\left(\mathbf{w}^{\mathsf T}\mathbf{x}_i+b\right)=1.
$$

Under this normalization, an SVM selects the separator that maximizes the minimum geometric distance to the training observations.

The two margin boundaries are

$$
\mathbf{w}^{\mathsf T}\mathbf{x} + b = 1
$$

and

$$
\mathbf{w}^{\mathsf T}\mathbf{x} + b = -1.
$$

The distance between them is

$$
\frac{2}{\lVert\mathbf{w}\rVert}.
$$

Therefore, under canonical normalization, maximizing the margin is equivalent to minimizing $\lVert\mathbf{w}\rVert$ (conventionally $\tfrac{1}{2}\lVert\mathbf{w}\rVert^2$) [@cortes1995svm].

In an exact hard-margin solution, support vectors lie on a margin boundary and satisfy $y_i f(\mathbf{x}_i)=1$. In a soft-margin solution, support vectors are the observations with nonzero dual coefficients: they may lie on the margin, inside it, or on the wrong side of the decision boundary. Thus, “support vector” is not synonymous with “misclassified observation.” Observations with zero dual coefficients do not directly enter the fitted decision function.

::: {.callout-note}
## Illustrations are not evaluations

The decision-boundary plots in this chapter fit models to all displayed observations to explain geometry. They are pedagogical training-set views, not estimates of predictive performance. The complete workflow later in the chapter uses cross-validation and an untouched test set.
:::

## Why feature scaling matters

SVMs are sensitive to differences in feature scales because the margin depends on geometric distances. A feature measured over a wide numerical range may dominate another feature whose values occupy a much narrower range.

The next example compares the decision boundary before and after standardization.

In [ ]:
#| label: fig-svm-scaling
#| fig-cap: Effect of feature scaling on a linear SVM.
#| code-fold: true
#| code-summary: Show code

X_scale = np.array([
    [1.0, 50],
    [5.0, 20],
    [3.0, 80],
    [5.0, 60]
])

y_scale = np.array([0, 0, 1, 1])

unscaled_model = SVC(kernel="linear", C=100)
unscaled_model.fit(X_scale, y_scale)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_scale)

scaled_model = SVC(kernel="linear", C=100)
scaled_model.fit(X_scaled, y_scale)

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4))

DecisionBoundaryDisplay.from_estimator(
    unscaled_model,
    X_scale,
    response_method="decision_function",
    plot_method="contour",
    levels=[-1, 0, 1],
    linestyles=["--", "-", "--"],
    ax=axes[0]
)

axes[0].scatter(X_scale[:, 0], X_scale[:, 1], c=y_scale)
axes[0].set_title("Without scaling")
axes[0].set_xlabel("$x_1$")
axes[0].set_ylabel("$x_2$")

DecisionBoundaryDisplay.from_estimator(
    scaled_model,
    X_scaled,
    response_method="decision_function",
    plot_method="contour",
    levels=[-1, 0, 1],
    linestyles=["--", "-", "--"],
    ax=axes[1]
)

axes[1].scatter(X_scaled[:, 0], X_scaled[:, 1], c=y_scale)
axes[1].set_title("After standardization")
axes[1].set_xlabel("Standardized $x_1$")
axes[1].set_ylabel("Standardized $x_2$")

plt.tight_layout()
plt.show()

::: {.callout-warning}
## Scale the features

For SVMs, scaling should normally be part of the modeling pipeline. The scaler must be fitted only on the training data to prevent data leakage.
:::

A robust Scikit-Learn workflow is:

In [ ]:
#| label: linear-svc-pipeline

svm_clf = make_pipeline(
    StandardScaler(),
    LinearSVC(C=1, loss="hinge", dual=True, random_state=42)
)

svm_clf.fit(X_margin, y_margin)

## Hard-margin classification

A hard-margin classifier requires every training observation to be correctly classified and to remain outside the margin:

$$
y_i\left(\mathbf{w}^{\mathsf T}\mathbf{x}_i+b\right) \geq 1,
\qquad i=1,\ldots,n.
$$

The optimization problem is

$$
\min_{\mathbf{w},b}
\frac{1}{2}\lVert\mathbf{w}\rVert^2
$$

subject to

$$
y_i\left(\mathbf{w}^{\mathsf T}\mathbf{x}_i+b\right) \geq 1.
$$

## Limitations of Hard-Margin Classification

Hard-margin SVM assumes that the training data are perfectly linearly separable. In practice, this assumption is often unrealistic because datasets frequently contain measurement errors, mislabeled observations, or naturally overlapping classes.

The following example illustrates two situations where hard-margin classification performs poorly.

In [ ]:
#| label: fig-hard-margin-limitations
#| fig-cap: Hard-margin infeasibility and the sensitivity of a near hard-margin approximation to an influential observation.
#| code-fold: true
#| code-summary: Show code

# Two artificial outliers
X_outliers = np.array([
    [3.4, 1.3],
    [3.2, 0.8]
])

y_outliers = np.array([0, 0])

# Dataset where perfect separation is impossible
X_impossible = np.vstack([
    X_margin,
    X_outliers[:1]
])

y_impossible = np.concatenate([
    y_margin,
    y_outliers[:1]
])

# Dataset with one influential outlier
X_sensitive = np.vstack([
    X_margin,
    X_outliers[1:]
])

y_sensitive = np.concatenate([
    y_margin,
    y_outliers[1:]
])

# A moderate finite C illustrates a close soft-margin approximation; SVC does
# not expose an exact hard-margin mode.
hard_margin_svm = SVC(
    kernel="linear",
    C=100
)

hard_margin_svm.fit(
    X_sensitive,
    y_sensitive
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10,4),
    sharey=True
)

# -------------------------------------------------
# Left panel
# -------------------------------------------------

ax = axes[0]

ax.plot(
    X_impossible[y_impossible==1,0],
    X_impossible[y_impossible==1,1],
    "bs",
    label="Iris versicolor"
)

ax.plot(
    X_impossible[y_impossible==0,0],
    X_impossible[y_impossible==0,1],
    "yo",
    label="Iris setosa"
)

ax.annotate(
    "Outlier",
    xy=X_outliers[0],
    xytext=(2.5,1.7),
    arrowprops=dict(arrowstyle="->")
)

ax.text(
    0.3,
    1.0,
    "Impossible!",
    fontsize=18,
    color="red"
)

ax.set_title("Perfect separation is impossible")
ax.set_xlabel("Petal length (cm)")
ax.set_ylabel("Petal width (cm)")
ax.set_xlim(0,5.5)
ax.set_ylim(0,2)
ax.grid(alpha=0.3)

# -------------------------------------------------
# Right panel
# -------------------------------------------------

ax = axes[1]

plot_svc_decision_boundary(
    hard_margin_svm,
    xmin=0,
    xmax=5.5,
    ax=ax
)

ax.plot(
    X_sensitive[y_sensitive==1,0],
    X_sensitive[y_sensitive==1,1],
    "bs"
)

ax.plot(
    X_sensitive[y_sensitive==0,0],
    X_sensitive[y_sensitive==0,1],
    "yo"
)

ax.annotate(
    "Outlier",
    xy=X_outliers[1],
    xytext=(3.2,0.1),
    arrowprops=dict(arrowstyle="->")
)

ax.set_title("Decision boundary dominated by one outlier")
ax.set_xlabel("Petal length (cm)")
ax.set_xlim(0,5.5)
ax.set_ylim(0,2)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

The left panel illustrates a situation where the classes are no longer perfectly linearly separable because of a single mislabeled observation. In this case, a hard-margin SVM cannot find a separating hyperplane, regardless of how it is positioned.

The right panel shows a different but equally important limitation. Although perfect separation is still possible, a nearly hard-margin fit can rotate substantially in response to one influential observation. The plotted `SVC(C=100)` is still a soft-margin estimator, used here as a documented finite-`C` approximation because Scikit-Learn's `SVC` does not provide an exact hard-margin mode [@scikitLearn2025].

These examples highlight why hard-margin classification is rarely used in practice. Real-world datasets almost always contain noise, measurement errors, or overlapping classes. Instead, Support Vector Machines generally employ **soft-margin classification**, which allows a small number of margin violations in exchange for a decision boundary that is much more robust.

## Soft-margin classification

Soft-margin classification allows some observations to fall inside the margin or even on the wrong side of the decision boundary. It introduces slack variables $\xi_i \geq 0$:

$$
y_i\left(\mathbf{w}^{\mathsf T}\mathbf{x}_i+b\right)
\geq 1-\xi_i.
$$

The corresponding objective is

$$
\min_{\mathbf{w},b,\boldsymbol{\xi}}
\frac{1}{2}\lVert\mathbf{w}\rVert^2
+
C\sum_{i=1}^{n}\xi_i.
$$

The hyperparameter $C$ controls the trade-off between a wide margin and margin violations:

- **large `C`** penalizes violations strongly and usually produces a narrower margin;
- **small `C`** tolerates more violations and usually produces a wider, more regularized margin.

### Visualizing the effect of `C`

In [ ]:
#| label: fig-svm-c
#| fig-cap: Effect of the regularization parameter C on the SVM margin.
#| code-fold: true
#| code-summary: Show code

iris = load_iris()
X_c = iris.data[:, (2, 3)]
y_c = (iris.target == 2).astype(int)

models = [
    make_pipeline(StandardScaler(), SVC(kernel="linear", C=0.1)),
    make_pipeline(StandardScaler(), SVC(kernel="linear", C=100))
]

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4))

for model, ax, c_value in zip(models, axes, [0.1, 100]):
    model.fit(X_c, y_c)

    DecisionBoundaryDisplay.from_estimator(
        model,
        X_c,
        response_method="decision_function",
        plot_method="contour",
        levels=[-1, 0, 1],
        linestyles=["--", "-", "--"],
        ax=ax
    )

    ax.scatter(X_c[y_c == 0, 0], X_c[y_c == 0, 1], marker="o")
    ax.scatter(X_c[y_c == 1, 0], X_c[y_c == 1, 1], marker="s")
    ax.set_title(f"C = {c_value}")
    ax.set_xlabel("Petal length (cm)")
    ax.set_ylabel("Petal width (cm)")

plt.tight_layout()
plt.show()

A very large `C` may reduce training errors but create a model with high variance. A smaller `C` generally increases regularization and may improve generalization.

## Hinge loss

The hinge loss for observation $i$ is

$$
L_i =
\max\left(
0,\,
1-y_i f(\mathbf{x}_i)
\right),
$$

where

$$
f(\mathbf{x}_i)=\mathbf{w}^{\mathsf T}\mathbf{x}_i+b.
$$

The value

$$
y_i f(\mathbf{x}_i)
$$

is the signed functional margin.

Three cases are possible:

- if $y_i f(\mathbf{x}_i)\geq 1$, the observation is correctly classified and on or outside the margin, so the loss is zero;
- if $0<y_i f(\mathbf{x}_i)<1$, it is correctly classified but inside the margin;
- if $y_i f(\mathbf{x}_i)=0$, it lies exactly on the decision boundary and the class assignment is a tie determined by the implementation's convention;
- if $y_i f(\mathbf{x}_i)<0$, it is misclassified.

In [ ]:
#| label: fig-hinge-loss
#| fig-cap: Hinge loss as a function of the signed margin.
#| code-fold: true
#| code-summary: Show code

signed_margin = np.linspace(-2, 3, 500)
hinge_loss = np.maximum(0, 1 - signed_margin)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(signed_margin, hinge_loss)
ax.axvline(0, linestyle="--")
ax.axvline(1, linestyle="--")
ax.set_xlabel(r"Signed margin $y f(\mathbf{x})$")
ax.set_ylabel("Hinge loss")
ax.set_title("Hinge loss")
plt.show()

::: {.callout-important}
## SVM scores are not probabilities

By default, SVM classifiers produce decision scores rather than class probabilities. `SVC(probability=True)` can estimate probabilities using an additional calibration procedure, but this increases training time.
:::

## Nonlinear SVM classification

A linear boundary is insufficient when the classes are not linearly separable in the original feature space. SVMs address this problem in two main ways:

1. explicitly create nonlinear features;
2. use a kernel that computes inner products in an implicit feature space.

In [ ]:
#| label: fig-feature-space-transformation
#| fig-cap: A nonlinear classification problem becomes linearly separable after mapping the observations to a higher-dimensional feature space.
#| code-fold: true
#| code-summary: Show code

# Original one-dimensional observations
X_original = np.linspace(-4, 4, 9).reshape(-1, 1)

# Feature mapping
X_transformed = np.c_[
    X_original,
    X_original ** 2
]

# Binary classes
y = np.array([0, 0, 1, 1, 1, 1, 1, 0, 0])

fig, axes = plt.subplots(
    1,
    2,
    figsize=(9.5, 4)
)

# ----------------------------------------------------
# Original feature space
# ----------------------------------------------------

ax = axes[0]

ax.grid(alpha=0.3)
ax.axhline(0, linewidth=1)

ax.plot(
    X_original[y == 0],
    np.zeros(np.sum(y == 0)),
    "s",
    label="Class 0"
)

ax.plot(
    X_original[y == 1],
    np.zeros(np.sum(y == 1)),
    "^",
    label="Class 1"
)

ax.set_xlabel(r"$x_1$")
ax.set_xlim(-4.5, 4.5)
ax.set_ylim(-0.2, 0.2)
ax.set_yticks([])
ax.set_title("Original feature space")

# ----------------------------------------------------
# Higher-dimensional feature space
# ----------------------------------------------------

ax = axes[1]

ax.grid(alpha=0.3)

ax.axhline(0, linewidth=1)
ax.axvline(0, linewidth=1)

ax.plot(
    X_transformed[y == 0, 0],
    X_transformed[y == 0, 1],
    "s",
    label="Class 0"
)

ax.plot(
    X_transformed[y == 1, 0],
    X_transformed[y == 1, 1],
    "^",
    label="Class 1"
)

# Linear separator

ax.plot(
    [-4.5, 4.5],
    [6.5, 6.5],
    "--",
    linewidth=2.5,
    label="Linear separator"
)

ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2=x_1^2$")
ax.set_xlim(-4.5, 4.5)
ax.set_ylim(-1, 17)

ax.set_title("Transformed feature space")

plt.tight_layout()
plt.show()

Consider the one-dimensional dataset shown in the left panel of Figure @fig-feature-space-transformation. No linear classifier can perfectly separate the two classes because every possible threshold leaves observations from both classes on the same side.

Now suppose that we create a new feature,

$$
x_2 = x_1^2.
$$

The observations are then represented in the two-dimensional feature space $(x_1,x_2)$ shown in the right panel. In this transformed space, the classes become linearly separable.

This simple example illustrates a fundamental idea in machine learning: a problem that is nonlinear in the original feature space may become linear after an appropriate feature transformation. Explicitly constructing these transformations is feasible for simple mappings such as $x_1^2$, but quickly becomes impractical when thousands or millions of transformed features are required. Support Vector Machines overcome this limitation through the **kernel trick**, which computes inner products in the transformed feature space without explicitly generating all transformed features.

### Polynomial features

A simple strategy is to transform the original inputs into polynomial features and then fit a linear SVM.

In [ ]:
#| label: polynomial-feature-svm

X_moons, y_moons = make_moons(
    n_samples=200,
    noise=0.15,
    random_state=42
)

polynomial_svm = make_pipeline(
    PolynomialFeatures(degree=3, include_bias=False),
    StandardScaler(),
    LinearSVC(C=10, loss="hinge", max_iter=20_000, random_state=42)
)

polynomial_svm.fit(X_moons, y_moons)

In [ ]:
#| label: fig-polynomial-features
#| fig-cap: Linear SVM fitted after a third-degree polynomial feature transformation.
#| code-fold: true
#| code-summary: Show code

fig, ax = plt.subplots(figsize=(7, 5))

DecisionBoundaryDisplay.from_estimator(
    polynomial_svm,
    X_moons,
    response_method="decision_function",
    plot_method="contourf",
    alpha=0.25,
    ax=ax
)

ax.scatter(
    X_moons[y_moons == 0, 0],
    X_moons[y_moons == 0, 1],
    marker="o",
    label="Class 0"
)

ax.scatter(
    X_moons[y_moons == 1, 0],
    X_moons[y_moons == 1, 1],
    marker="s",
    label="Class 1"
)

ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.legend()
plt.show()

Explicit polynomial expansion is intuitive, but the number of generated features can increase quickly with the degree and the original number of predictors.

## The kernel trick

A kernel computes a similarity measure between two observations:

$$
K(\mathbf{x},\mathbf{x}')
=
\phi(\mathbf{x})^{\mathsf T}\phi(\mathbf{x}'),
$$

where $\phi(\cdot)$ represents a potentially high-dimensional feature transformation.

The kernel trick makes it possible to work with the inner products in that transformed space without explicitly computing all transformed features.

Not every similarity function is a valid SVM kernel. For any finite set of observations, a valid real-valued kernel must produce a symmetric, positive-semidefinite Gram matrix. This condition ensures that the kernel corresponds to an inner product in some feature space. Scikit-Learn accepts built-in kernels, a precomputed Gram matrix, or a callable, but accepting a callable does not guarantee that the supplied function satisfies this mathematical requirement [@scikitLearn2025].

Common kernels include:

### Linear kernel

$$
K(\mathbf{x},\mathbf{x}')
=
\mathbf{x}^{\mathsf T}\mathbf{x}'.
$$

### Polynomial kernel

$$
K(\mathbf{x},\mathbf{x}')
=
\left(
\gamma\,\mathbf{x}^{\mathsf T}\mathbf{x}'
+r
\right)^d,
$$

where:

- $d$ is the polynomial degree;
- $\gamma$ controls the influence of the inner product;
- $r$, represented by `coef0` in Scikit-Learn, controls the contribution of lower- and higher-order terms.

### Radial basis function kernel

$$
K(\mathbf{x},\mathbf{x}')
=
\exp\left(
-\gamma
\lVert\mathbf{x}-\mathbf{x}'\rVert^2
\right).
$$

The RBF kernel is one of the most commonly used nonlinear kernels.

## Polynomial kernel

In [ ]:
#| label: polynomial-kernel-svm

poly_kernel_svm = make_pipeline(
    StandardScaler(),
    SVC(
        kernel="poly",
        degree=3,
        coef0=1,
        C=5
    )
)

poly_kernel_svm.fit(X_moons, y_moons)

In [ ]:
#| label: fig-polynomial-kernel
#| fig-cap: Decision boundary produced by a third-degree polynomial kernel.
#| code-fold: true

fig, ax = plt.subplots(figsize=(7, 5))

DecisionBoundaryDisplay.from_estimator(
    poly_kernel_svm,
    X_moons,
    response_method="decision_function",
    plot_method="contourf",
    alpha=0.25,
    ax=ax
)

ax.scatter(X_moons[y_moons == 0, 0], X_moons[y_moons == 0, 1], marker="o")
ax.scatter(X_moons[y_moons == 1, 0], X_moons[y_moons == 1, 1], marker="s")

ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
plt.show()

Polynomial-kernel tuning is not monotonic. Raising `degree` does not guarantee a more useful boundary or better training score because `degree`, `gamma`, `coef0`, and `C` jointly determine the feature-space geometry and regularization. Similarly, simply reversing one adjustment does not guarantee that overfitting will decrease. Tune these parameters together with cross-validation, preferably over small, computationally plausible grids.

## Similarity features

A nonlinear transformation can also be constructed by measuring the similarity between an observation and selected landmarks.

Using a Gaussian radial basis function centered at landmark $\boldsymbol{\ell}$,

$$
\phi_{\boldsymbol{\ell}}(\mathbf{x})
=
\exp\left(
-\gamma
\lVert\mathbf{x}-\boldsymbol{\ell}\rVert^2
\right).
$$

A small distance from the landmark produces a value close to 1, whereas a large distance produces a value close to 0.

In [ ]:
#| label: fig-rbf-similarity
#| fig-cap: 'Gaussian RBF similarity features. Left: observations in the original one-dimensional space and their similarity to two landmarks. Right: observations in the transformed feature space.'
#| code-fold: true
#| code-summary: Show code

def gaussian_rbf(X, landmark, gamma):
    """
    Compute the Gaussian radial basis function similarity
    between each observation in X and a landmark.
    """
    return np.exp(
        -gamma * np.linalg.norm(X - landmark, axis=1) ** 2
    )


# One-dimensional observations
X_1d = np.linspace(-4, 4, 9).reshape(-1, 1)

# Binary class labels
y_rbf = np.array([0, 0, 1, 1, 1, 1, 1, 0, 0])

# Landmarks and RBF parameter
landmark_1 = -2
landmark_2 = 1
gamma = 0.3

# Continuous grid used to draw the similarity functions
x_grid = np.linspace(-4.5, 4.5, 500).reshape(-1, 1)

similarity_landmark_1 = gaussian_rbf(
    x_grid,
    landmark_1,
    gamma
)

similarity_landmark_2 = gaussian_rbf(
    x_grid,
    landmark_2,
    gamma
)

# Transform the original observations into two similarity features
X_transformed = np.c_[
    gaussian_rbf(X_1d, landmark_1, gamma),
    gaussian_rbf(X_1d, landmark_2, gamma)
]

fig, axes = plt.subplots(
    1,
    2,
    figsize=(9.5, 4)
)

# ---------------------------------------------------------
# Left panel: original one-dimensional feature space
# ---------------------------------------------------------

ax = axes[0]

ax.axhline(
    y=0,
    linewidth=1
)

ax.scatter(
    [landmark_1, landmark_2],
    [0, 0],
    s=150,
    alpha=0.5,
    label="Landmarks"
)

ax.plot(
    X_1d[y_rbf == 0, 0],
    np.zeros(np.sum(y_rbf == 0)),
    "s",
    label="Class 0"
)

ax.plot(
    X_1d[y_rbf == 1, 0],
    np.zeros(np.sum(y_rbf == 1)),
    "^",
    label="Class 1"
)

ax.plot(
    x_grid[:, 0],
    similarity_landmark_1,
    "--",
    label=fr"Similarity to $\ell_1={landmark_1}$"
)

ax.plot(
    x_grid[:, 0],
    similarity_landmark_2,
    ":",
    label=fr"Similarity to $\ell_2={landmark_2}$"
)

ax.annotate(
    r"$\mathbf{x}$",
    xy=(X_1d[3, 0], 0),
    xytext=(-0.5, 0.22),
    ha="center",
    arrowprops={
        "arrowstyle": "->"
    },
    fontsize=14
)

ax.text(
    landmark_1,
    0.90,
    r"$\ell_1$",
    ha="center",
    fontsize=14
)

ax.text(
    landmark_2,
    0.90,
    r"$\ell_2$",
    ha="center",
    fontsize=14
)

ax.set_xlabel(r"Original feature $x_1$")
ax.set_ylabel("Similarity")
ax.set_xlim(-4.5, 4.5)
ax.set_ylim(-0.1, 1.1)
ax.set_yticks([0, 0.25, 0.5, 0.75, 1])
ax.grid(alpha=0.3)
ax.legend(fontsize=8, loc="upper right")


# ---------------------------------------------------------
# Right panel: transformed similarity-feature space
# ---------------------------------------------------------

ax = axes[1]

ax.axhline(
    y=0,
    linewidth=1
)

ax.axvline(
    x=0,
    linewidth=1
)

ax.plot(
    X_transformed[y_rbf == 0, 0],
    X_transformed[y_rbf == 0, 1],
    "s",
    label="Class 0"
)

ax.plot(
    X_transformed[y_rbf == 1, 0],
    X_transformed[y_rbf == 1, 1],
    "^",
    label="Class 1"
)

ax.annotate(
    r"$\phi(\mathbf{x})$",
    xy=(
        X_transformed[3, 0],
        X_transformed[3, 1]
    ),
    xytext=(0.68, 0.52),
    ha="center",
    arrowprops={
        "arrowstyle": "->"
    },
    fontsize=14
)

# Illustrative linear decision boundary
x_boundary = np.array([-0.1, 1.1])
y_boundary = 0.51 - 0.61 * x_boundary

ax.plot(
    x_boundary,
    y_boundary,
    "--",
    linewidth=2.5,
    label="Linear decision boundary"
)

ax.set_xlabel(
    r"$x_2=\phi_{\ell_1}(\mathbf{x})$"
)

ax.set_ylabel(
    r"$x_3=\phi_{\ell_2}(\mathbf{x})$"
)

ax.set_xlim(-0.1, 1.1)
ax.set_ylim(-0.1, 1.1)
ax.grid(alpha=0.3)
ax.legend(fontsize=8, loc="upper right")

plt.tight_layout()
plt.show()

The left panel shows the original one-dimensional observations and
their Gaussian RBF similarity to two landmarks located at
$\ell_1=-2$ and $\ell_2=1$.

Each original observation is transformed into two new features:

$$
x_2=\phi_{\ell_1}(\mathbf{x})=\exp\left(-\gamma\lVert \mathbf{x}-\ell_1\rVert^2\right),
$$

and

$$
x_3=\phi_{\ell_2}(\mathbf{x})=\exp\left(-\gamma\lVert \mathbf{x}-\ell_2\rVert^2\right).
$$

The right panel represents the observations in the transformed
feature space $(x_2,x_3)$. Although the classes are not linearly
separable in the original one-dimensional space, the RBF
transformation makes it possible to separate them using a linear
decision boundary.

Increasing `gamma` makes each similarity curve narrower. Decreasing it makes the curve wider.

## Gaussian RBF kernel

An RBF SVM can learn complex nonlinear boundaries without explicitly creating one feature per landmark.

In [ ]:
#| label: rbf-kernel-svm

rbf_svm = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf", gamma=5, C=1)
)

rbf_svm.fit(X_moons, y_moons)

### Interpreting `gamma`

The hyperparameter `gamma` controls how rapidly the similarity decreases with distance:

- **large `gamma`** gives each training observation a narrow region of influence and can produce an irregular boundary;
- **small `gamma`** gives observations a broader region of influence and produces a smoother boundary.

Thus, `gamma` behaves as an inverse measure of the radius of influence.

### Interaction between `C` and `gamma`

The following figure compares several combinations.

In [ ]:
#| label: fig-rbf-grid
#| fig-cap: Interaction between C and gamma for an RBF SVM.
#| code-fold: true
#| code-summary: Show code

parameter_combinations = [
    (0.1, 0.1),
    (100, 0.1),
    (0.1, 5),
    (100, 5)
]

fig, axes = plt.subplots(2, 2, figsize=(9.5, 8))

for ax, (c_value, gamma_value) in zip(
    axes.ravel(),
    parameter_combinations
):
    model = make_pipeline(
        StandardScaler(),
        SVC(
            kernel="rbf",
            C=c_value,
            gamma=gamma_value
        )
    )

    model.fit(X_moons, y_moons)

    DecisionBoundaryDisplay.from_estimator(
        model,
        X_moons,
        response_method="decision_function",
        plot_method="contourf",
        alpha=0.25,
        ax=ax
    )

    ax.scatter(
        X_moons[y_moons == 0, 0],
        X_moons[y_moons == 0, 1],
        marker="o"
    )

    ax.scatter(
        X_moons[y_moons == 1, 0],
        X_moons[y_moons == 1, 1],
        marker="s"
    )

    ax.set_title(
        fr"$C={c_value}$, $\gamma={gamma_value}$"
    )

    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")

plt.tight_layout()
plt.show()

A large `C` combined with a large `gamma` can fit highly localized patterns and may overfit. A small `C` combined with a small `gamma` produces a smoother and more regularized model.

::: {.callout-tip}
## Practical tuning strategy

For an RBF SVM:

1. standardize all numerical predictors;
2. begin with logarithmic grids for `C` and `gamma`;
3. evaluate combinations with cross-validation;
4. inspect both predictive performance and the gap between training and validation scores.
:::

## Multiclass classification

SVMs are fundamentally binary classifiers, but Scikit-Learn extends them to multiclass problems.

`SVC` uses a one-versus-one strategy internally. For $K$ classes, it trains

$$
\frac{K(K-1)}{2}
$$

binary classifiers.

`LinearSVC` uses a one-versus-rest strategy by default, fitting one classifier for each class.

In [ ]:
#| label: multiclass-svm

iris = load_iris()

multiclass_svm = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf", C=1, gamma="scale")
)

multiclass_svm.fit(iris.data, iris.target)

prediction = multiclass_svm.predict(
    iris.data[[0]]
)

prediction

## SVM regression

SVMs can also be used for regression. The objective is reversed relative to classification: the model attempts to place as many observations as possible inside a tube extending $\varepsilon$ above and $\varepsilon$ below the regression function. Its total vertical width is therefore $2\varepsilon$.

For prediction function

$$
f(\mathbf{x})
=
\mathbf{w}^{\mathsf T}\mathbf{x}+b,
$$

the $\varepsilon$-insensitive loss is

$$
L_{\varepsilon}
=
\max\left(
0,\,
|y-f(\mathbf{x})|-\varepsilon
\right).
$$

Errors smaller than $\varepsilon$ are ignored.

### Linear SVR

In [ ]:
#| label: fig-linear-svr
#| fig-cap: Linear support vector regression with two values of epsilon.
#| code-fold: true

rng = np.random.default_rng(42)

X_reg = 2 * rng.random((80, 1))
y_reg = 4 + 3 * X_reg[:, 0] + rng.normal(0, 0.8, 80)

epsilon_values = [0.2, 1.0]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

x_plot = np.linspace(0, 2, 300).reshape(-1, 1)

for epsilon, ax in zip(epsilon_values, axes):
    model = make_pipeline(
        StandardScaler(),
        LinearSVR(
            epsilon=epsilon,
            C=1,
            dual="auto",
            max_iter=20_000,
            random_state=42
        )
    )

    model.fit(X_reg, y_reg)
    y_plot = model.predict(x_plot)

    ax.scatter(X_reg[:, 0], y_reg)
    ax.plot(x_plot[:, 0], y_plot)
    ax.plot(x_plot[:, 0], y_plot + epsilon, linestyle="--")
    ax.plot(x_plot[:, 0], y_plot - epsilon, linestyle="--")
    ax.set_title(fr"$\varepsilon={epsilon}$")
    ax.set_xlabel("$x$")
    ax.set_ylabel("$y$")

plt.tight_layout()
plt.show()

The model is called $\varepsilon$-insensitive because movements of training observations within the tube do not affect the loss. In Scikit-Learn, `epsilon` and the residual penalty controlled by `C` use the target's numerical units, so changing a target from dollars to thousands of dollars changes their practical meaning [@scikitLearn2025].

When target magnitudes make tuning awkward, `TransformedTargetRegressor` can standardize `y` during fitting and automatically invert that transformation for predictions:

In [ ]:
#| label: scaled-target-svr

scaled_target_svr = TransformedTargetRegressor(
    regressor=make_pipeline(
        StandardScaler(),
        SVR(kernel="rbf", C=1, epsilon=0.1, gamma="scale")
    ),
    transformer=StandardScaler()
)

scaled_target_svr.fit(X_reg, y_reg)
scaled_target_predictions = scaled_target_svr.predict(X_reg[:3])
scaled_target_predictions

Here, `epsilon=0.1` is measured in standardized-target units during fitting, while `predict()` returns values on the original target scale. In a parameter search, the nested name is `regressor__svr__epsilon`. Fit the target transformer only through the wrapper inside each training or cross-validation fold; never standardize all target values before splitting.

### Nonlinear SVR

A kernelized SVR can model nonlinear relationships.

In [ ]:
#| label: fig-polynomial-svr
#| fig-cap: Polynomial-kernel SVR with different values of C.
#| code-fold: true

rng = np.random.default_rng(42)

X_quad = 2 * rng.random((100, 1)) - 1
y_quad = (
    0.2
    + 0.1 * X_quad[:, 0]
    + 0.5 * X_quad[:, 0] ** 2
    + rng.normal(0, 0.08, 100)
)

models = [
    SVR(kernel="poly", degree=2, C=100, epsilon=0.1, coef0=1),
    SVR(kernel="poly", degree=2, C=0.01, epsilon=0.1, coef0=1)
]

x_plot = np.linspace(-1, 1, 300).reshape(-1, 1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for model, ax, c_value in zip(models, axes, [100, 0.01]):
    model.fit(X_quad, y_quad)
    y_plot = model.predict(x_plot)

    ax.scatter(X_quad[:, 0], y_quad)
    ax.plot(x_plot[:, 0], y_plot)
    ax.plot(x_plot[:, 0], y_plot + model.epsilon, linestyle="--")
    ax.plot(x_plot[:, 0], y_plot - model.epsilon, linestyle="--")
    ax.set_title(fr"Polynomial SVR, $C={c_value}$")
    ax.set_xlabel("$x$")
    ax.set_ylabel("$y$")

plt.tight_layout()
plt.show()

A larger `C` reduces regularization and attempts to fit the training observations more closely. A smaller `C` produces a smoother model.

## Novelty detection with `OneClassSVM`

`OneClassSVM` estimates a boundary around a reference distribution without requiring class labels. It is most appropriate for **novelty detection**: training data should be clean, or at least contain very little contamination, and future observations are flagged when they fall outside the learned region. This differs from supervised classification and from the harder task of fitting directly to a substantially contaminated dataset [@scikitLearn2025].

In [ ]:
#| label: one-class-svm

rng = np.random.default_rng(42)
X_reference = rng.normal(loc=0, scale=1, size=(160, 2))
X_future = np.array([
    [0.2, -0.3],
    [-0.8, 0.5],
    [3.8, 4.2],
    [-4.0, 3.5],
])

novelty_detector = make_pipeline(
    StandardScaler(),
    OneClassSVM(kernel="rbf", gamma="scale", nu=0.05)
)
novelty_detector.fit(X_reference)

novelty_labels = novelty_detector.predict(X_future)
novelty_scores = novelty_detector.decision_function(X_future)

list(zip(X_future.tolist(), novelty_labels.tolist(), novelty_scores.round(3)))

Predictions are `+1` for observations inside the estimated region and `-1` for flagged novelties; larger decision scores indicate greater compatibility with the reference distribution. The parameter `nu` is not the expected novelty rate. Under the one-class SVM optimization assumptions, it is an upper bound on the fraction of training errors and a lower bound on the fraction of support vectors. Results can be sensitive to scaling, `gamma`, and `nu`, so select them using domain knowledge or labeled validation anomalies when available. A low score is evidence of distributional difference, not proof that an observation is erroneous, dangerous, or from a particular alternative class.

## Computational considerations

The main Scikit-Learn SVM estimators have different computational characteristics.

| Estimator | Main use | Kernel support | Typical behavior |
|---|---|---:|---|
| `LinearSVC` | Linear classification | No | Scales comparatively well to large datasets |
| `SVC(kernel="linear")` | Linear classification | Yes | Useful when support vectors are required |
| `SVC(kernel="rbf")` | Nonlinear classification | Yes | Powerful but expensive for large datasets |
| `LinearSVR` | Linear regression | No | Suitable for larger datasets |
| `SVR` | Nonlinear regression | Yes | Can become slow as sample size increases |

Kernel SVMs usually require substantial computation as the number of observations grows. Their training time can increase rapidly because the algorithm works with pairwise relationships between observations.

## Practical modeling workflow

A sound SVM workflow keeps preprocessing, model selection, and final evaluation separate [@scikitLearn2025]:

1. split the data into training and test sets;
2. place preprocessing and SVM estimation in one pipeline;
3. standardize numerical variables;
4. select a kernel according to the expected boundary complexity;
5. tune `C`, and for RBF models tune `gamma`;
6. use cross-validation for model selection;
7. evaluate the final model once on untouched test data;
8. inspect class imbalance and choose suitable metrics.

The following complete example uses only generated data. The stratified split preserves class proportions, and the test set is set aside immediately.

In [ ]:
#| label: svm-workflow-split

X_workflow, y_workflow = make_moons(
    n_samples=500,
    noise=0.25,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X_workflow,
    y_workflow,
    test_size=0.20,
    stratify=y_workflow,
    random_state=42
)

Scaling belongs inside `Pipeline`, so every cross-validation training fold fits its own scaler. The logarithmic grids cover multiplicative changes in `C` and `gamma`; a linear candidate is included as a lower-complexity comparison.

In [ ]:
#| label: svm-workflow-search

svm_pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("svc", SVC())
])

parameter_grid = [
    {
        "svc__kernel": ["linear"],
        "svc__C": np.logspace(-2, 2, 5),
    },
    {
        "svc__kernel": ["rbf"],
        "svc__C": np.logspace(-2, 2, 5),
        "svc__gamma": np.logspace(-2, 1, 4),
    },
]

stratified_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

svm_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=parameter_grid,
    scoring="balanced_accuracy",
    cv=stratified_cv,
    return_train_score=True,
    n_jobs=-1,
    refit=True
)
svm_search.fit(X_train, y_train)

`cv_results_` contains the training and validation score for every candidate. Reporting both helps distinguish insufficient flexibility from excessive flexibility without consulting the test set.

In [ ]:
#| label: svm-workflow-cv-results

cv_results = svm_search.cv_results_
top_indices = np.argsort(cv_results["rank_test_score"])[:5]

cv_summary = [
    {
        "parameters": {
            name: value.item() if isinstance(value, np.generic) else value
            for name, value in cv_results["params"][index].items()
        },
        "mean_train_balanced_accuracy": float(
            round(cv_results["mean_train_score"][index], 3)
        ),
        "mean_validation_balanced_accuracy": float(
            round(cv_results["mean_test_score"][index], 3)
        ),
        "validation_standard_deviation": float(
            round(cv_results["std_test_score"][index], 3)
        ),
    }
    for index in top_indices
]

cv_summary

In [ ]:
#| label: svm-workflow-diagnosis

best_index = svm_search.best_index_
best_train_score = cv_results["mean_train_score"][best_index]
best_validation_score = cv_results["mean_test_score"][best_index]
generalization_gap = best_train_score - best_validation_score

if best_validation_score < 0.75:
    cv_diagnosis = "Possible underfitting: both validation performance and model utility are low."
elif generalization_gap > 0.08:
    cv_diagnosis = "Possible overfitting: training performance substantially exceeds validation performance."
else:
    cv_diagnosis = "No strong underfitting or overfitting signal in the cross-validation scores."

{
    "best_parameters": {
        name: value.item() if isinstance(value, np.generic) else value
        for name, value in svm_search.best_params_.items()
    },
    "mean_train_balanced_accuracy": float(round(best_train_score, 3)),
    "mean_validation_balanced_accuracy": float(round(best_validation_score, 3)),
    "train_validation_gap": float(round(generalization_gap, 3)),
    "diagnosis": cv_diagnosis,
}

The thresholds in this diagnostic are teaching heuristics, not universal rules. Adequate performance depends on the application, and fold variability should also be considered. Only after all choices are fixed do we use the held-out set, exactly once:

In [ ]:
#| label: svm-workflow-final-test

y_test_prediction = svm_search.predict(X_test)

final_test_metrics = {
    "accuracy": round(accuracy_score(y_test, y_test_prediction), 3),
    "balanced_accuracy": float(
        round(balanced_accuracy_score(y_test, y_test_prediction), 3)
    ),
    "f1": round(f1_score(y_test, y_test_prediction), 3),
    "confusion_matrix": confusion_matrix(
        y_test, y_test_prediction
    ).tolist(),
}

final_test_metrics

Accuracy summarizes the overall hit rate, balanced accuracy gives equal weight to the two class recalls, F1 combines precision and recall for class 1, and the confusion matrix exposes the error counts. These test metrics estimate final performance; they must not trigger another round of tuning. If they are unexpectedly weak, report that result and design a new evaluation rather than repeatedly adapting to this test set.

## Common mistakes

::: {.callout-warning}
## Data leakage

Do not standardize the complete dataset before splitting it. Fit the scaler only through a pipeline trained on the training data.
:::

::: {.callout-warning}
## Unscaled predictors

Using predictors with very different scales may distort the geometry of the problem and produce a poor decision boundary.
:::

::: {.callout-warning}
## Interpreting `C` incorrectly

In Scikit-Learn, a larger `C` means **less regularization**, while a smaller `C` means **more regularization**.
:::

::: {.callout-warning}
## Tuning on the test set

The test set should not be used to select `C`, `gamma`, the kernel, or any preprocessing decision.
:::

## Chapter summary

Under canonical normalization, a hard-margin SVM finds the widest-margin separator for linearly separable training data. A soft-margin SVM instead balances margin width against violations through `C`. Its support vectors have nonzero dual coefficients and may be on the margin, inside it, or misclassified.

Nonlinear SVMs use valid positive-semidefinite kernels to represent complex boundaries without explicitly generating all transformed features. Polynomial-kernel behavior depends jointly on `degree`, `gamma`, `coef0`, and `C`; it is not monotonic in degree. In an RBF model, `gamma` controls the radius of influence, while `C` controls the penalty for margin violations.

SVM regression ignores errors within $\varepsilon$ of the fitted function, giving a tube of total width $2\varepsilon$. Target scaling changes the units of `epsilon` and `C`; `TransformedTargetRegressor` can manage that transformation safely. `OneClassSVM` supports novelty detection when its reference training sample is mostly clean, but a novelty flag does not explain why an observation differs.

In all SVM applications, scaling and hyperparameter selection must occur inside the training process. Use a pipeline, tune on cross-validation folds, diagnose train-validation gaps, and evaluate the selected model once on an untouched test set. Decision-boundary plots fitted to displayed data explain model geometry but do not estimate generalization performance.

## Exercises

1. Fit finite-`C` linear SVM classifiers to two Iris classes using several modest values of `C`. Compare margin violations and support-vector counts, and explain why none of these fits is an exact hard-margin estimator.

2. Generate a moons dataset with a higher noise level. Compare a polynomial-feature SVM, a polynomial-kernel SVM, and an RBF SVM using the same stratified cross-validation folds. Do not use the test set to choose among them.

3. For an RBF classifier, create logarithmic grids for `C` and `gamma` inside a leakage-safe pipeline. Report mean training and validation scores, their gap, and one final held-out balanced accuracy.

4. Explain why increasing RBF `gamma` may lead to overfitting and why increasing polynomial `degree` need not change performance monotonically.

5. Compare `LinearSVC` and `SVC(kernel="linear")` on the same standardized training folds. Examine balanced accuracy, training time, support-vector availability, and the effect of `C`.

6. Fit an `SVR` model to a nonlinear regression dataset. Compare tuning with the target in original units and with `TransformedTargetRegressor`; interpret `epsilon` in each case.

7. Train `OneClassSVM` on clean simulated reference observations, then test increasingly shifted observations. Vary `nu` and `gamma`, and explain why the fraction flagged in future data is not guaranteed to equal `nu`.

8. For the selected classifier in the workflow, use `cv_results_` to identify one clearly underfit candidate and one candidate that shows a larger train-validation gap. Justify both diagnoses without consulting the test set.